# Phase 1 — Target Preparation

This notebook walks through the complete Phase 1 pipeline for preparing
PdG and E3G target structures for the NanoLFA-Design pipeline.

**What this notebook does:**
1. Generates 3D conformers from SMILES for each target analyte
2. Energy-minimizes conformers with MMFF94 force field
3. Computes molecular properties relevant to hapten–antibody recognition
4. Models the linker attachment point and simulates carrier-protein occlusion
5. Maps the epitope surface (SASA-based) to identify nanobody-accessible regions
6. Extracts pharmacophore features on the epitope surface
7. Processes the cross-reactivity panel (structural analogs)
8. Exports all structures to SDF/PDB for downstream AlphaFold 3 input

**Prerequisites:**
```bash
conda activate nanolfa
pip install -e .  # install nanolfa package
```

In [ ]:
import logging
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from omegaconf import OmegaConf

# NanoLFA modules
from nanolfa.utils.chemistry import (
    generate_conformers,
    export_molecule,
    compute_molecular_properties,
    compute_sasa,
    map_epitope_surface,
    extract_pharmacophore_features,
    identify_linker_attachment_atom,
    model_linker_occlusion,
    build_epitope_surface_map,
    prepare_target,
)

# RDKit for visualization
from rdkit import Chem
from rdkit.Chem import Draw, AllChem

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# Output directory
OUTPUT_DIR = Path('../data/targets')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('All imports successful.')

## 1. Load Target Configurations

Each target is defined in a YAML config with SMILES, carrier protein info,
linker chemistry, and a cross-reactivity panel.

In [ ]:
# Load both target configs
pdg_cfg = OmegaConf.load('../configs/targets/pdg.yaml')
e3g_cfg = OmegaConf.load('../configs/targets/e3g.yaml')

targets = {
    'pdg': OmegaConf.to_container(pdg_cfg.target, resolve=True),
    'e3g': OmegaConf.to_container(e3g_cfg.target, resolve=True),
}

for name, cfg in targets.items():
    print(f"\n{'=' * 50}")
    print(f"Target: {cfg['full_name']} ({name.upper()})")
    print(f"  SMILES: {cfg['smiles'][:60]}...")
    print(f"  MW: {cfg['mol_weight']} Da")
    print(f"  Carrier: {cfg['carrier_protein']}")
    print(f"  Linker: {cfg['linker_chemistry']}")
    print(f"  Cross-reactivity panel: {len(cfg['cross_reactivity_panel'])} compounds")

## 2. Visualize Target Structures (2D)

Before generating 3D conformers, let's look at the 2D structures of our
primary analytes and their cross-reactivity panels.

In [ ]:
# Draw PdG and E3G side by side
pdg_mol = Chem.MolFromSmiles(targets['pdg']['smiles'])
e3g_mol = Chem.MolFromSmiles(targets['e3g']['smiles'])

img = Draw.MolsToGridImage(
    [pdg_mol, e3g_mol],
    molsPerRow=2,
    subImgSize=(500, 400),
    legends=['PdG (Pregnanediol-3-glucuronide)', 'E3G (Estrone-3-glucuronide)'],
)
img

In [ ]:
# Draw the PdG cross-reactivity panel
pdg_panel_mols = []
pdg_panel_names = []
for compound in targets['pdg']['cross_reactivity_panel']:
    mol = Chem.MolFromSmiles(compound['smiles'])
    if mol:
        pdg_panel_mols.append(mol)
        pdg_panel_names.append(f"{compound['name']}\n(max CR: {compound['max_cross_reactivity_pct']}%)")

print(f"PdG cross-reactivity panel: {len(pdg_panel_mols)} compounds")
Draw.MolsToGridImage(
    pdg_panel_mols,
    molsPerRow=3,
    subImgSize=(400, 300),
    legends=pdg_panel_names,
)

## 3. Generate 3D Conformers

For each target, we generate 200 conformers using ETKDG v3, then energy-minimize
with MMFF94 and select the lowest-energy conformer as the representative structure.

Why this matters for nanobody design: the 3D shape of the steroid ring system
determines binding pocket geometry. Steroid A/B ring puckering (chair vs. boat)
and the orientation of the glucuronide moiety define the pharmacophore that the
nanobody CDR loops must complement.

In [ ]:
# Generate conformers for PdG
pdg_result = generate_conformers(
    smiles=targets['pdg']['smiles'],
    name='pdg',
    n_conformers=200,
    force_field='MMFF94',
    random_seed=42,
)

print(f"\nPdG conformer generation:")
print(f"  Generated: {pdg_result.num_conformers_generated}")
print(f"  After pruning: {pdg_result.num_conformers_after_pruning}")
print(f"  Best energy: {pdg_result.best_energy_kcal:.2f} kcal/mol")
print(f"  Best conformer ID: {pdg_result.best_conformer_id}")

In [ ]:
# Generate conformers for E3G
e3g_result = generate_conformers(
    smiles=targets['e3g']['smiles'],
    name='e3g',
    n_conformers=200,
    force_field='MMFF94',
    random_seed=42,
)

print(f"\nE3G conformer generation:")
print(f"  Generated: {e3g_result.num_conformers_generated}")
print(f"  After pruning: {e3g_result.num_conformers_after_pruning}")
print(f"  Best energy: {e3g_result.best_energy_kcal:.2f} kcal/mol")

## 4. Molecular Properties

Compare the physicochemical properties of PdG and E3G. These affect
how the nanobody CDR loops should be designed:
- **LogP**: hydrophobicity — drives the balance of aromatic vs. polar CDR contacts
- **H-bond donors/acceptors**: defines the hydrogen bonding network in the binding pocket
- **Aromatic rings**: E3G has one (A-ring), enabling pi-stacking with Trp/Tyr CDR residues
- **Fraction Csp3**: PdG is more sp3-rich (more 3D), E3G is partially aromatic (flatter)

In [ ]:
pdg_props = compute_molecular_properties(pdg_result.mol)
e3g_props = compute_molecular_properties(e3g_result.mol)

# Display as a comparison table
print(f"{'Property':<22} {'PdG':>10} {'E3G':>10}")
print('-' * 44)
for key in pdg_props:
    label = key.replace('_', ' ').title()
    print(f"{label:<22} {pdg_props[key]:>10.2f} {e3g_props[key]:>10.2f}")

In [ ]:
# Radar chart comparing key properties
categories = ['LogP', 'HBD', 'HBA', 'TPSA', 'Rotatable\nBonds', 'Aromatic\nRings']
pdg_vals = [pdg_props['logp'], pdg_props['hbd'], pdg_props['hba'],
            pdg_props['tpsa'] / 30, pdg_props['rotatable_bonds'], pdg_props['aromatic_rings']]
e3g_vals = [e3g_props['logp'], e3g_props['hbd'], e3g_props['hba'],
            e3g_props['tpsa'] / 30, e3g_props['rotatable_bonds'], e3g_props['aromatic_rings']]

angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
pdg_vals_plot = pdg_vals + [pdg_vals[0]]
e3g_vals_plot = e3g_vals + [e3g_vals[0]]
angles += [angles[0]]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
ax.plot(angles, pdg_vals_plot, 'o-', linewidth=2, label='PdG', color='#2196F3')
ax.fill(angles, pdg_vals_plot, alpha=0.15, color='#2196F3')
ax.plot(angles, e3g_vals_plot, 's-', linewidth=2, label='E3G', color='#E91E63')
ax.fill(angles, e3g_vals_plot, alpha=0.15, color='#E91E63')
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=11)
ax.set_title('Molecular Property Comparison: PdG vs E3G', size=14, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.15, 1.1))
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'property_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Epitope Surface Mapping

The key question for nanobody design: **which parts of the hapten are accessible**
**for binding when it's conjugated to a carrier protein?**

Both PdG and E3G are conjugated to BSA via carbodiimide (EDC) coupling through
the glucuronide carboxylic acid group. This means:
- The glucuronide COOH is consumed by the linker (buried)
- The glucuronide hydroxyl groups are partially occluded
- The steroid ring system is maximally exposed (this is the epitope)

We model this by computing SASA of the free hapten, then applying distance-dependent
occlusion around the linker attachment point.

In [ ]:
# Build epitope surface map for PdG
pdg_epitope = build_epitope_surface_map(
    pdg_result,
    linker_chemistry='carbodiimide',
    linker_length_atoms=6,
    sasa_threshold=5.0,
)

print(f"PdG Epitope Surface:")
print(f"  Total SASA: {pdg_epitope.total_sasa:.1f} A^2")
print(f"  Epitope SASA: {pdg_epitope.epitope_sasa:.1f} A^2")
print(f"  Epitope fraction: {pdg_epitope.epitope_fraction:.1%}")
print(f"  Exposed atoms: {len(pdg_epitope.epitope_atom_indices)}")
print(f"  Buried atoms: {len(pdg_epitope.buried_atom_indices)}")

In [ ]:
# Build epitope surface map for E3G
e3g_epitope = build_epitope_surface_map(
    e3g_result,
    linker_chemistry='carbodiimide',
    linker_length_atoms=6,
    sasa_threshold=5.0,
)

print(f"E3G Epitope Surface:")
print(f"  Total SASA: {e3g_epitope.total_sasa:.1f} A^2")
print(f"  Epitope SASA: {e3g_epitope.epitope_sasa:.1f} A^2")
print(f"  Epitope fraction: {e3g_epitope.epitope_fraction:.1%}")
print(f"  Exposed atoms: {len(e3g_epitope.epitope_atom_indices)}")
print(f"  Buried atoms: {len(e3g_epitope.buried_atom_indices)}")

In [ ]:
# Visualize SASA distribution for both targets
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, epitope) in zip(axes, [('PdG', pdg_epitope), ('E3G', e3g_epitope)]):
    sasa_values = [a.sasa for a in epitope.atom_sasa]
    colors = ['#E91E63' if a.is_epitope else '#9E9E9E' for a in epitope.atom_sasa]

    ax.bar(range(len(sasa_values)), sasa_values, color=colors, width=0.8)
    ax.axhline(y=5.0, color='black', linestyle='--', linewidth=1, alpha=0.5, label='Threshold (5 A^2)')
    ax.set_xlabel('Heavy Atom Index', size=11)
    ax.set_ylabel('SASA (A^2)', size=11)
    ax.set_title(f'{name} — Per-Atom SASA\n(red = epitope-exposed, grey = buried)', size=12)
    ax.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'sasa_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Pharmacophore Features

Pharmacophore features on the epitope surface tell us what kinds of interactions
the nanobody CDR loops should make. This directly informs the amino acid bias
we apply during ProteinMPNN sequence design in Phase 2.

In [ ]:
# Compare pharmacophore features between PdG and E3G
feature_types = ['donor', 'acceptor', 'hydrophobic', 'aromatic', 'positive', 'negative']

for name, epitope in [('PdG', pdg_epitope), ('E3G', e3g_epitope)]:
    print(f"\n{name} Pharmacophore Features (epitope-accessible):")
    for ft in feature_types:
        total = len([f for f in epitope.pharmacophore_features if f.feature_type == ft])
        accessible = len([
            f for f in epitope.pharmacophore_features
            if f.feature_type == ft and f.is_epitope_accessible
        ])
        bar = '#' * accessible
        print(f"  {ft:<14} {accessible:>2}/{total:<2}  {bar}")

In [ ]:
# Bar chart of epitope-accessible pharmacophore features
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(feature_types))
width = 0.35

pdg_counts = [
    len([f for f in pdg_epitope.pharmacophore_features
         if f.feature_type == ft and f.is_epitope_accessible])
    for ft in feature_types
]
e3g_counts = [
    len([f for f in e3g_epitope.pharmacophore_features
         if f.feature_type == ft and f.is_epitope_accessible])
    for ft in feature_types
]

ax.bar(x - width / 2, pdg_counts, width, label='PdG', color='#2196F3', alpha=0.8)
ax.bar(x + width / 2, e3g_counts, width, label='E3G', color='#E91E63', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([ft.title() for ft in feature_types], size=11)
ax.set_ylabel('Count', size=12)
ax.set_title('Epitope-Accessible Pharmacophore Features', size=14)
ax.legend(fontsize=12)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pharmacophore_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Export Structures

Export the best conformers to SDF (for AlphaFold 3 ligand input) and PDB formats.

In [ ]:
# Export PdG
pdg_dir = OUTPUT_DIR / 'pdg' / 'structures'
pdg_result = export_molecule(pdg_result, pdg_dir, formats=('sdf', 'pdb'))
print(f"PdG exported to: {pdg_result.output_sdf}")

# Export E3G
e3g_dir = OUTPUT_DIR / 'e3g' / 'structures'
e3g_result = export_molecule(e3g_result, e3g_dir, formats=('sdf', 'pdb'))
print(f"E3G exported to: {e3g_result.output_sdf}")

## 8. Full Pipeline Run

The `prepare_target()` function runs the entire Phase 1 pipeline for a single
target, including cross-reactivity panel processing. This is what the CLI
script `scripts/prepare_targets.py` calls internally.

In [ ]:
# Run the full pipeline for PdG (including cross-reactivity panel)
pdg_full = prepare_target(
    target_config=targets['pdg'],
    output_dir=OUTPUT_DIR,
    n_conformers=200,
    force_field='MMFF94',
    random_seed=42,
)

print(f"\nPdG Phase 1 complete:")
print(f"  Cross-reactants processed: {len(pdg_full.cross_reactants)}")
for cr in pdg_full.cross_reactants:
    print(f"    {cr.name}: {cr.best_energy_kcal:.2f} kcal/mol")
print(f"  Summary saved to: {pdg_full.output_dir / 'preparation_summary.json'}")

In [ ]:
# Run the full pipeline for E3G
e3g_full = prepare_target(
    target_config=targets['e3g'],
    output_dir=OUTPUT_DIR,
    n_conformers=200,
    force_field='MMFF94',
    random_seed=42,
)

print(f"\nE3G Phase 1 complete:")
print(f"  Cross-reactants processed: {len(e3g_full.cross_reactants)}")
for cr in e3g_full.cross_reactants:
    print(f"    {cr.name}: {cr.best_energy_kcal:.2f} kcal/mol")

## 9. Review Summary

Load and display the preparation summaries written by the pipeline.

In [ ]:
for target in ['pdg', 'e3g']:
    summary_path = OUTPUT_DIR / target / 'preparation_summary.json'
    if summary_path.exists():
        with open(summary_path) as f:
            summary = json.load(f)
        print(f"\n{'=' * 50}")
        print(f"Summary for {target.upper()}:")
        print(json.dumps(summary, indent=2))

## 10. Design Implications

Based on the epitope surface analysis, here are the key takeaways for
nanobody CDR design in Phase 2:

### PdG
- Steroid A/B rings are the primary epitope surface
- Multiple H-bond donors/acceptors on the steroid hydroxyl groups
- No aromatic rings — CDR contacts should favor H-bonding and van der Waals
- Glucuronide partially occluded by linker — CDR3 should not rely on glucuronide contacts
- Longer CDR3 (12–20 residues) recommended to form a deep pocket around the steroid

### E3G
- Aromatic A-ring is exposed — pi-stacking with Trp/Tyr in CDR3 is favorable
- D-ring ketone is a key H-bond acceptor
- Slightly smaller molecule — CDR3 can be shorter (10–18 residues)
- The aromatic character suggests boosting Trp (pi-stacking) and Arg (cation-pi + glucuronide salt bridge)

These observations are encoded in the target configs:
- `configs/targets/pdg.yaml` → CDR3 length 12–20, boost W/Y/F
- `configs/targets/e3g.yaml` → CDR3 length 10–18, boost W/Y/F/R

---

**Next step:** Phase 2 — Seed Nanobody Generation (notebook `02_seed_generation.ipynb`)